# biogas-pvrv-BN — run everything in Colab

This notebook clones the repository, installs the requirements and runs the two drivers. Runtime → *Change runtime type* → CPU is enough; a GPU brings nothing here.

1. Bayesian network: about one minute.
2. Pressure model, quick mode (first 30 days): a few minutes.
3. Pressure model, full year: roughly 20 minutes with numba (25 one-year simulations and 9 event scans).

In [ ]:
REPO = 'https://github.com/drinkgarlic/biogas-pvrv-BN.git'
import os
if not os.path.exists('biogas-pvrv-BN'):
    !git clone $REPO
%cd biogas-pvrv-BN
!pip install -q -r requirements.txt
import numba, numpy, pandas, matplotlib; print('numba', numba.__version__, '| numpy', numpy.__version__, '| pandas', pandas.__version__)

## 1. Bayesian network (nine cases, sensitivity, figures)

In [ ]:
!python colab/run_bayesian_network.py --v3

In [ ]:
import pandas as pd
r = pd.read_csv('bayesian_network/out/bn_onoff_9cases.csv')
r[r.variant == 'retry30'][['case', 'f_on', 'open_demand', 'open_stop', 'open_block', 'open_foam', 'open_wv', 'openings', 'vent_m3', 'ch4_kg']].round(3)

In [ ]:
from IPython.display import Image, display
for f in ['bayesian_network/fig/bn_onoff_9cases.png', 'bayesian_network/fig/bn_vs_pressure_onoff.png', 'bayesian_network/fig/bn_onoff_case1_weekly.png']:
    display(Image(f, width=900))

## 2. Pressure model — quick check (first 30 days)

The same workflow as the study, restricted to 30 days, to make sure the kernel compiles and every stage runs.

In [ ]:
!python colab/run_pressure_model.py --quick --days 30 --jobs 2

In [ ]:
import pandas as pd
pd.read_csv('pressure_model/out/onoff_summary.csv')[['tag', 'fill_mean', 'flare_starts', 'flare_fails', 'pv_episodes', 'pv_open_h', 'vent_m3', 'PT_max']].round(3)

## 3. Pressure model — the full year

About 15 minutes on a Colab CPU runtime with numba. Reduce `--jobs` to 1 if memory runs out (each year simulation needs ~1 GB).

In [ ]:
!python colab/run_pressure_model.py --jobs 2

In [ ]:
import pandas as pd
print(pd.read_csv('pressure_model/out/summary_base.csv').round(3).T)
print(pd.read_csv('pressure_model/out/onoff_summary.csv')[['tag', 'pv_episodes', 'pv_cycles', 'pv_open_h', 'vent_m3', 'ch4_kg', 'fill_mean', 'PT_max']].round(3))

In [ ]:
from IPython.display import Image, display
for f in ['pressure_model/fig/base_case1_year.png', 'pressure_model/fig/stop_scan.png', 'pressure_model/fig/onoff_9cases_fill_pv.png', 'pressure_model/fig/onoff_case1_episode.png']:
    display(Image(f, width=900))

## 4. Download the outputs

In [ ]:
!zip -q -r outputs.zip pressure_model/out pressure_model/fig bayesian_network/out bayesian_network/fig
from google.colab import files
files.download('outputs.zip')

## 5. (Optional) keep the outputs in Google Drive

The Colab runtime is wiped when it disconnects. Run this cell to copy the outputs to `My Drive/biogas-pvrv-BN/`.

In [ ]:
from google.colab import drive
import shutil, os
drive.mount('/content/drive')
dst = '/content/drive/MyDrive/biogas-pvrv-BN'
os.makedirs(dst, exist_ok=True)
for d in ['pressure_model/out', 'pressure_model/fig', 'bayesian_network/out', 'bayesian_network/fig']:
    if os.path.isdir(d):
        shutil.copytree(d, os.path.join(dst, d), dirs_exist_ok=True)
print('copied to', dst)